In [1]:
# D13 — Branch C: Normalised Markdown conversion

In [2]:
# ============================================================
# 0. Imports and frozen experimental configuration
# ============================================================

!pip -q install openpyxl

from google.colab import files
from pathlib import Path
from datetime import datetime
from collections import Counter
from openpyxl import load_workbook

import hashlib
import json
import platform
import re
import sys
import unicodedata

import pandas as pd

DOCUMENT_ID = "D13"
DOCUMENT_NAME = "Eurostat — Unemployment rates by country of birth"

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

SOURCE_FORMAT = ".xlsx"

EXPECTED_SOURCE_SHA256 = (
    "13f6d031c0e5c888d5632ad20853d16e603f462cdf3b58c0dd40057824d10f9a"
)

EXPECTED_BRANCH_B_REPRESENTATION_SHA256 = (
    "0577c86a97bf8ce594464fdc771d7d2dd9fb32e917bf259cf706f1d4a9e98c69"
)

EXPECTED_WORKSHEET_COUNT = 16
EXPECTED_NON_EMPTY_CELL_COUNT = 12978

SUMMARY_SHEET = "Summary"

EXPECTED_SHEETS = [
    "Summary"
] + [
    f"Sheet {number}"
    for number in range(1, 16)
]

SELECTED_SHEETS = [
    "Sheet 1",
    "Sheet 2",
    "Sheet 3",
    "Sheet 4",
    "Sheet 5"
]

SELECTED_GEOGRAPHIES = [
    "European Union - 27 countries (from 2020)",
    "Belgium",
    "Germany",
    "Spain",
    "Portugal"
]

SELECTED_YEARS = [
    2020,
    2022,
    2024
]

# ------------------------------------------------------------
# Frozen Stage 1 diagnostics.
# NOT disclosed to the model.
# ------------------------------------------------------------

EXPECTED_RECORD_COUNT = (
    len(SELECTED_SHEETS)
    * len(SELECTED_GEOGRAPHIES)
    * len(SELECTED_YEARS)
)

EXPECTED_YEAR_COUNTS = {
    str(year):
        len(SELECTED_SHEETS)
        * len(SELECTED_GEOGRAPHIES)
    for year in SELECTED_YEARS
}

EXPECTED_CATEGORY = "Labour market time series"
EXPECTED_TOPIC = "Unemployment rate by country of birth"
EXPECTED_UNIT = "percent"

EXPECTED_CATEGORY_COUNTS = {
    EXPECTED_CATEGORY:
        EXPECTED_RECORD_COUNT
}

EXPECTED_FLAG_COUNTS = {
    "d": 10,
    "b": 5,
    "u": 2
}

EXPECTED_FLAGGED_RECORDS = 17

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]

OUTPUT_DIR = Path("outputs_D13_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARENT_CHECK_PATH = (
    OUTPUT_DIR / "D13_branch_C_parent_B_equivalence_check.json"
)

NORMALISATION_CHECK_PATH = (
    OUTPUT_DIR / "D13_branch_C_normalisation_check.json"
)

REPRESENTATION_PATH = (
    OUTPUT_DIR / "D13_branch_C_normalised_markdown.md"
)

REPRESENTATION_METADATA_PATH = (
    OUTPUT_DIR / "D13_branch_C_representation_metadata.json"
)

SELECTION_SCOPE_PATH = (
    OUTPUT_DIR / "D13_branch_C_selection_scope.csv"
)

PROMPT_PATH = (
    OUTPUT_DIR / "D13_branch_C_prompt.txt"
)

EXPERIMENT_METADATA_PRE_PATH = (
    OUTPUT_DIR / "D13_branch_C_experiment_metadata_pre.json"
)

PRECHECK_PATH = (
    OUTPUT_DIR / "D13_branch_C_pre_extraction_check.json"
)

RAW_RESPONSE_PATH = (
    OUTPUT_DIR / "D13_branch_C_raw_response.txt"
)

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR / "D13_branch_C_parsed_extraction.json"
)

STRUCTURE_CHECK_PATH = (
    OUTPUT_DIR / "D13_branch_C_structure_check.json"
)

SCOPE_CHECK_PATH = (
    OUTPUT_DIR / "D13_branch_C_scope_check.csv"
)

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR / "D13_branch_C_experiment_metadata.json"
)

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR / "D13_branch_C_experiment_summary.json"
)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Parent branch:", PARENT_BRANCH)
print("Expected source worksheets:", EXPECTED_WORKSHEET_COUNT)
print("Expected represented non-empty cells:", EXPECTED_NON_EMPTY_CELL_COUNT)


Document: D13
Branch: C
Parent branch: B
Expected source worksheets: 16
Expected represented non-empty cells: 12978


In [3]:
# ============================================================
# 1. Upload original D13 XLSX and frozen Branch B artefacts
# ============================================================
#
# Upload exactly:
#   1) original D13 XLSX
#   2) D13_branch_B_structural_markdown.md
#   3) D13_branch_B_conversion_integrity.json
#
# Branch C does NOT regenerate the Branch B representation.
# ============================================================

uploaded = files.upload()
names = list(uploaded.keys())

xlsx_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".xlsx")
]

md_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".md")
]

json_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".json")
]

if (
    len(xlsx_files) != 1
    or len(md_files) != 1
    or len(json_files) != 1
):
    raise ValueError(
        "Upload exactly one D13 XLSX, one frozen Branch B structural "
        "Markdown file, and one Branch B conversion-integrity JSON file."
    )

SOURCE_PATH = xlsx_files[0]
BRANCH_B_REPRESENTATION_PATH = md_files[0]
BRANCH_B_CHECK_PATH = json_files[0]

print("Source:", SOURCE_PATH.name)
print("Branch B representation:", BRANCH_B_REPRESENTATION_PATH.name)
print("Branch B integrity:", BRANCH_B_CHECK_PATH.name)


Saving D13_branch_B_structural_markdown.md to D13_branch_B_structural_markdown.md
Saving D13_branch_B_conversion_integrity.json to D13_branch_B_conversion_integrity.json
Saving D13 - Unemployment rates by country of birth_2026.xlsx to D13 - Unemployment rates by country of birth_2026.xlsx
Source: D13 - Unemployment rates by country of birth_2026.xlsx
Branch B representation: D13_branch_B_structural_markdown.md
Branch B integrity: D13_branch_B_conversion_integrity.json


In [4]:
# ============================================================
# 2. Verify frozen source identity, workbook structure and B provenance
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(chunk_size),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def sha256_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def clean_text(value):
    if value is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

SOURCE_HASH_MATCH = (
    SOURCE_SHA256
    == EXPECTED_SOURCE_SHA256
)

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded D13 workbook does not match the frozen Stage 1 source identity."
    )


workbook_values = load_workbook(
    SOURCE_PATH,
    data_only=True,
    read_only=False
)

workbook_formulas = load_workbook(
    SOURCE_PATH,
    data_only=False,
    read_only=False
)

sheet_names = workbook_values.sheetnames

worksheet_count_valid = (
    len(sheet_names)
    == EXPECTED_WORKSHEET_COUNT
)

worksheet_names_valid = (
    sheet_names
    == EXPECTED_SHEETS
)

formula_cell_count = 0
non_empty_cell_count = 0

for sheet_name in sheet_names:

    ws_values = workbook_values[
        sheet_name
    ]

    ws_formulas = workbook_formulas[
        sheet_name
    ]

    for row in ws_values.iter_rows():
        for cell in row:
            if cell.value is not None:
                non_empty_cell_count += 1

    for row in ws_formulas.iter_rows():
        for cell in row:
            if (
                isinstance(cell.value, str)
                and cell.value.startswith("=")
            ):
                formula_cell_count += 1


summary_text = " ".join(
    clean_text(cell.value)
    for row
    in workbook_values[
        SUMMARY_SHEET
    ].iter_rows()
    for cell in row
    if cell.value is not None
)

summary_metadata_checks = {
    "dataset_identifier":
        "lfsa_urgacob"
        in summary_text,

    "dataset_title":
        "Unemployment rates by country of birth"
        in summary_text,

    "annual_frequency":
        "Annual"
        in summary_text,

    "percentage_unit":
        "Percentage"
        in summary_text,

    "age_class":
        "From 15 to 74 years"
        in summary_text
}

summary_metadata_valid = all(
    summary_metadata_checks.values()
)


with open(
    BRANCH_B_CHECK_PATH,
    "r",
    encoding="utf-8"
) as f:
    branch_b_check = json.load(f)


if branch_b_check.get(
    "document_id"
) != DOCUMENT_ID:
    raise ValueError(
        "Branch B conversion-integrity artefact belongs to another document."
    )

if branch_b_check.get(
    "branch"
) != "B":
    raise ValueError(
        "Uploaded conversion-integrity artefact is not from Branch B."
    )

if branch_b_check.get(
    "source_sha256"
) != SOURCE_SHA256:
    raise ValueError(
        "Branch B conversion-integrity artefact refers to another D13 source."
    )

if not branch_b_check.get(
    "conversion_integrity_passed",
    False
):
    raise ValueError(
        "The frozen D13 Branch B representation did not pass conversion integrity."
    )


SOURCE_B_MARKDOWN = (
    BRANCH_B_REPRESENTATION_PATH.read_text(
        encoding="utf-8"
    )
)

if not SOURCE_B_MARKDOWN.strip():
    raise ValueError(
        "Uploaded Branch B structural Markdown is empty."
    )


UPLOADED_BRANCH_B_SHA256 = sha256_text(
    SOURCE_B_MARKDOWN
)

BRANCH_B_HASH_MATCH = (
    UPLOADED_BRANCH_B_SHA256
    == EXPECTED_BRANCH_B_REPRESENTATION_SHA256
)

if not BRANCH_B_HASH_MATCH:
    raise ValueError(
        "Uploaded Branch B Markdown does not match the frozen final "
        "D13 Branch B representation SHA-256."
    )


SOURCE_INTEGRITY_VALID = all([
    SOURCE_HASH_MATCH,
    worksheet_count_valid,
    worksheet_names_valid,
    summary_metadata_valid,
    formula_cell_count == 0,
    non_empty_cell_count
        == EXPECTED_NON_EMPTY_CELL_COUNT
])

if not SOURCE_INTEGRITY_VALID:
    raise ValueError(
        "D13 source workbook integrity verification failed."
    )


print("Frozen source identity verified.")
print("Source workbook structure verified.")
print("Branch B conversion provenance verified.")
print("Frozen Branch B SHA-256 verified:", BRANCH_B_HASH_MATCH)
print("Non-empty source cells:", non_empty_cell_count)


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Frozen source identity verified.
Source workbook structure verified.
Branch B conversion provenance verified.
Frozen Branch B SHA-256 verified: True
Non-empty source cells: 12978


In [5]:
# ============================================================
# 3. Derive fixed extraction-scope identities for diagnostics only
# ============================================================
#
# These workbook-derived locations/flags are NOT supplied to the model
# and are NOT used to construct the Branch C representation.
# ============================================================

def find_year_columns(worksheet):

    year_columns = {}

    for column_index in range(
        1,
        worksheet.max_column + 1
    ):

        value = worksheet.cell(
            row=11,
            column=column_index
        ).value

        if (
            isinstance(value, int)
            and not isinstance(value, bool)
        ):
            year_columns[value] = (
                column_index
            )

        elif (
            isinstance(value, str)
            and re.fullmatch(
                r"20\d{2}",
                value.strip()
            )
        ):
            year_columns[
                int(value.strip())
            ] = column_index

    return year_columns


def find_geography_rows(worksheet):

    geography_rows = {}

    for row_index in range(
        13,
        worksheet.max_row + 1
    ):

        label = clean_text(
            worksheet.cell(
                row=row_index,
                column=1
            ).value
        )

        if label:
            geography_rows[
                label
            ] = row_index

    return geography_rows


scope_rows = []

for sheet_name in SELECTED_SHEETS:

    worksheet = workbook_values[
        sheet_name
    ]

    year_columns = find_year_columns(
        worksheet
    )

    geography_rows = find_geography_rows(
        worksheet
    )

    for geography in SELECTED_GEOGRAPHIES:

        if geography not in geography_rows:
            raise ValueError(
                f"{geography!r} was not found in {sheet_name}."
            )

        row_index = geography_rows[
            geography
        ]

        for year in SELECTED_YEARS:

            if year not in year_columns:
                raise ValueError(
                    f"Year {year} was not found in {sheet_name}."
                )

            value_column = year_columns[
                year
            ]

            value_cell = worksheet.cell(
                row=row_index,
                column=value_column
            )

            flag_cell = worksheet.cell(
                row=row_index,
                column=value_column + 1
            )

            flag_value = (
                clean_text(
                    flag_cell.value
                )
                if flag_cell.value
                is not None
                else ""
            )

            scope_rows.append({
                "Sheet":
                    sheet_name,

                "Geography":
                    geography,

                "Year":
                    year,

                "Expected Value Cell":
                    value_cell.coordinate,

                "Expected Flag Cell":
                    flag_cell.coordinate,

                "Expected Source Location":
                    (
                        f"{sheet_name}, "
                        f"cell {value_cell.coordinate}"
                    ),

                "Expected Statistical Flag":
                    flag_value
            })


selection_scope_df = pd.DataFrame(
    scope_rows
)

expected_source_locations = (
    selection_scope_df[
        "Expected Source Location"
    ].tolist()
)

expected_source_location_set = set(
    expected_source_locations
)

scope_structure_valid = all([
    len(selection_scope_df)
        == EXPECTED_RECORD_COUNT,

    len(expected_source_location_set)
        == EXPECTED_RECORD_COUNT
])

if not scope_structure_valid:
    raise ValueError(
        "D13 fixed extraction-scope structure is invalid."
    )


observed_scope_flag_counts = dict(
    Counter(
        flag
        for flag
        in selection_scope_df[
            "Expected Statistical Flag"
        ].tolist()
        if flag
    )
)

scope_flag_counts_valid = (
    observed_scope_flag_counts
    == EXPECTED_FLAG_COUNTS
)

scope_flagged_record_count = sum(
    bool(flag)
    for flag in selection_scope_df[
        "Expected Statistical Flag"
    ]
)

scope_flagged_record_count_valid = (
    scope_flagged_record_count
    == EXPECTED_FLAGGED_RECORDS
)


selection_scope_df.to_csv(
    SELECTION_SCOPE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Fixed selected observations:",
    len(selection_scope_df)
)

print(
    "Expected flagged observations:",
    scope_flagged_record_count
)

print(
    "Flag counts:",
    observed_scope_flag_counts
)


Fixed selected observations: 75
Expected flagged observations: 17
Flag counts: {'b': 5, 'd': 10, 'u': 2}


In [6]:
# ============================================================
# 4. Verify exact frozen Branch B parent representation
# ============================================================

WORKSHEET_HEADER_PATTERN = re.compile(
    r"^## Worksheet:\s*(.+?)\s*$"
)

CELL_LINE_PATTERN = re.compile(
    r"^-\s+([A-Z]+\d+)\s+\[([a-z]+)\]:\s+(.+)$"
)


def parse_structural_markdown(
    markdown_text
):
    parsed = []
    worksheet_order = []

    current_sheet = None

    for line in markdown_text.splitlines():

        worksheet_match = (
            WORKSHEET_HEADER_PATTERN.fullmatch(
                line.strip()
            )
        )

        if worksheet_match:

            current_sheet = (
                worksheet_match.group(1)
            )

            worksheet_order.append(
                current_sheet
            )

            continue

        cell_match = (
            CELL_LINE_PATTERN.fullmatch(
                line.strip()
            )
        )

        if (
            cell_match
            and current_sheet is not None
        ):

            coordinate = (
                cell_match.group(1)
            )

            value_type = (
                cell_match.group(2)
            )

            encoded_value = (
                cell_match.group(3)
            )

            try:
                decoded_value = json.loads(
                    encoded_value
                )
            except json.JSONDecodeError:
                raise ValueError(
                    "Could not decode frozen Branch B cell "
                    f"{current_sheet}!{coordinate}."
                )

            parsed.append({
                "Sheet":
                    current_sheet,

                "Cell":
                    coordinate,

                "Type":
                    value_type,

                "Encoded Value":
                    encoded_value,

                "Decoded Value":
                    decoded_value
            })

    return (
        parsed,
        worksheet_order
    )


(
    PARENT_CELL_RECORDS,
    PARENT_WORKSHEET_ORDER
) = parse_structural_markdown(
    SOURCE_B_MARKDOWN
)


parent_cell_count_valid = (
    len(PARENT_CELL_RECORDS)
    == EXPECTED_NON_EMPTY_CELL_COUNT
)

parent_worksheet_order_valid = (
    PARENT_WORKSHEET_ORDER
    == EXPECTED_SHEETS
)

parent_cell_keys = [
    (
        record["Sheet"],
        record["Cell"]
    )
    for record
    in PARENT_CELL_RECORDS
]

parent_cell_keys_unique = (
    len(parent_cell_keys)
    == len(
        set(parent_cell_keys)
    )
)

parent_scope_value_cells_present = all(
    (
        row["Sheet"],
        row["Expected Value Cell"]
    )
    in set(parent_cell_keys)
    for _, row
    in selection_scope_df.iterrows()
)

parent_scope_flag_cells_present_when_nonempty = all(
    (
        (
            row["Sheet"],
            row["Expected Flag Cell"]
        )
        in set(parent_cell_keys)
    )
    or
    (
        not row[
            "Expected Statistical Flag"
        ]
    )
    for _, row
    in selection_scope_df.iterrows()
)


PARENT_EQUIVALENCE_PASSED = bool(
    SOURCE_INTEGRITY_VALID
    and branch_b_check.get(
        "conversion_integrity_passed",
        False
    )
    and BRANCH_B_HASH_MATCH
    and parent_cell_count_valid
    and parent_worksheet_order_valid
    and parent_cell_keys_unique
    and parent_scope_value_cells_present
    and parent_scope_flag_cells_present_when_nonempty
)


parent_check = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "source_sha256":
        SOURCE_SHA256,

    "source_hash_matches_frozen_identity":
        SOURCE_HASH_MATCH,

    "branch_B_conversion_integrity_passed":
        bool(
            branch_b_check.get(
                "conversion_integrity_passed",
                False
            )
        ),

    "expected_frozen_branch_B_sha256":
        EXPECTED_BRANCH_B_REPRESENTATION_SHA256,

    "uploaded_branch_B_sha256":
        UPLOADED_BRANCH_B_SHA256,

    "uploaded_branch_B_hash_matches_frozen_parent":
        BRANCH_B_HASH_MATCH,

    "expected_worksheet_count":
        EXPECTED_WORKSHEET_COUNT,

    "parent_worksheet_count":
        len(
            PARENT_WORKSHEET_ORDER
        ),

    "parent_worksheet_order_valid":
        parent_worksheet_order_valid,

    "expected_non_empty_cell_count":
        EXPECTED_NON_EMPTY_CELL_COUNT,

    "parent_represented_cell_count":
        len(
            PARENT_CELL_RECORDS
        ),

    "parent_cell_count_valid":
        parent_cell_count_valid,

    "parent_cell_keys_unique":
        parent_cell_keys_unique,

    "parent_scope_value_cells_present":
        parent_scope_value_cells_present,

    "parent_scope_flag_cells_present_when_nonempty":
        parent_scope_flag_cells_present_when_nonempty,

    "parent_equivalence_method":
        (
            "Frozen Branch B representation SHA-256 + Branch B "
            "conversion-integrity provenance; Branch B is not regenerated"
        ),

    "branch_B_regeneration_attempted":
        False,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED
}


PARENT_CHECK_PATH.write_text(
    json.dumps(
        parent_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        parent_check,
        ensure_ascii=False,
        indent=2
    )
)


if not PARENT_EQUIVALENCE_PASSED:
    raise ValueError(
        "D13 Branch C parent-equivalence verification failed."
    )


{
  "document_id": "D13",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "13f6d031c0e5c888d5632ad20853d16e603f462cdf3b58c0dd40057824d10f9a",
  "source_hash_matches_frozen_identity": true,
  "branch_B_conversion_integrity_passed": true,
  "expected_frozen_branch_B_sha256": "0577c86a97bf8ce594464fdc771d7d2dd9fb32e917bf259cf706f1d4a9e98c69",
  "uploaded_branch_B_sha256": "0577c86a97bf8ce594464fdc771d7d2dd9fb32e917bf259cf706f1d4a9e98c69",
  "uploaded_branch_B_hash_matches_frozen_parent": true,
  "expected_worksheet_count": 16,
  "parent_worksheet_count": 16,
  "parent_worksheet_order_valid": true,
  "expected_non_empty_cell_count": 12978,
  "parent_represented_cell_count": 12978,
  "parent_cell_count_valid": true,
  "parent_cell_keys_unique": true,
  "parent_scope_value_cells_present": true,
  "parent_scope_flag_cells_present_when_nonempty": true,
  "parent_equivalence_method": "Frozen Branch B representation SHA-256 + Branch B conversion-integrity provenance; Branch B is not 

In [7]:
# ============================================================
# 5. Define deterministic Branch C normalisation
# ============================================================
#
# String cell values are normalised conservatively.
# Cell coordinates, worksheet order, source types and all non-string
# values remain unchanged.
#
# Allowed:
# - Unicode NFKC;
# - Unicode-space standardisation;
# - typographic apostrophe and quote standardisation;
# - dash/minus standardisation;
# - soft-hyphen removal;
# - internal whitespace collapse in string values;
# - line-ending standardisation.
#
# NOT applied:
# - worksheet/row/column/scope filtering;
# - cell reordering;
# - statistical-flag reconstruction;
# - semantic label remapping;
# - numeric calculation/rescaling/rounding;
# - unit conversion;
# - manual correction;
# - reference-guided repair.
# ============================================================

UNICODE_SPACE_CHARACTERS = [
    "\u00a0", "\u1680", "\u2000", "\u2001", "\u2002",
    "\u2003", "\u2004", "\u2005", "\u2006", "\u2007",
    "\u2008", "\u2009", "\u200a", "\u202f", "\u205f",
    "\u3000"
]

APOSTROPHE_REPLACEMENTS = {
    "’": "'",
    "‘": "'",
    "‛": "'",
    "´": "'",
    "`": "'"
}

QUOTE_REPLACEMENTS = {
    "“": '"',
    "”": '"'
}

DASH_REPLACEMENTS = {
    "‐": "-",
    "‑": "-",
    "‒": "-",
    "–": "-",
    "—": "-",
    "−": "-"
}


def normalise_string_value(
    value
):
    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(
            character,
            " "
        )

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    for source, target in QUOTE_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    text = text.replace(
        "\u00ad",
        ""
    )

    text = (
        text
        .replace("\r\n", "\n")
        .replace("\r", "\n")
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def normalise_non_cell_line(
    line
):
    text = unicodedata.normalize(
        "NFKC",
        line
    )

    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(
            character,
            " "
        )

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    for source, target in QUOTE_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    text = text.replace(
        "\u00ad",
        ""
    )

    return re.sub(
        r"[ \t\f\v]+",
        " ",
        text
    ).rstrip()


In [8]:
# ============================================================
# 6. Apply Branch C normalisation to the COMPLETE frozen B representation
# ============================================================

normalised_lines = []

for line in SOURCE_B_MARKDOWN.splitlines():

    cell_match = (
        CELL_LINE_PATTERN.fullmatch(
            line.strip()
        )
    )

    if cell_match:

        coordinate = (
            cell_match.group(1)
        )

        value_type = (
            cell_match.group(2)
        )

        encoded_value = (
            cell_match.group(3)
        )

        decoded_value = json.loads(
            encoded_value
        )

        if isinstance(
            decoded_value,
            str
        ):
            normalised_value = (
                normalise_string_value(
                    decoded_value
                )
            )

        else:
            normalised_value = (
                decoded_value
            )

        normalised_encoded_value = (
            json.dumps(
                normalised_value,
                ensure_ascii=False,
                allow_nan=False
            )
        )

        normalised_lines.append(
            f"- {coordinate} "
            f"[{value_type}]: "
            f"{normalised_encoded_value}"
        )

    else:

        normalised_lines.append(
            normalise_non_cell_line(
                line
            )
        )


NORMALISED_MARKDOWN = (
    "\n".join(
        normalised_lines
    )
    .rstrip()
    + "\n"
)


if not NORMALISED_MARKDOWN.strip():
    raise ValueError(
        "D13 Branch C normalisation produced an empty representation."
    )


print(
    "Branch B characters:",
    len(
        SOURCE_B_MARKDOWN
    )
)

print(
    "Branch C characters:",
    len(
        NORMALISED_MARKDOWN
    )
)

print(
    "Representation changed:",
    SOURCE_B_MARKDOWN
    != NORMALISED_MARKDOWN
)


Branch B characters: 281409
Branch C characters: 281376
Representation changed: True


In [9]:
# ============================================================
# 7. Verify Branch C normalisation integrity
# ============================================================
#
# IMPORTANT:
# This validates the COMPLETE 16-sheet representation.
# It does not validate only the 75 target records.
# ============================================================

(
    BRANCH_C_CELL_RECORDS,
    BRANCH_C_WORKSHEET_ORDER
) = parse_structural_markdown(
    NORMALISED_MARKDOWN
)


branch_c_cell_count_valid = (
    len(BRANCH_C_CELL_RECORDS)
    == EXPECTED_NON_EMPTY_CELL_COUNT
)

worksheet_order_preserved = (
    BRANCH_C_WORKSHEET_ORDER
    == PARENT_WORKSHEET_ORDER
    == EXPECTED_SHEETS
)

parent_signature = [
    (
        record["Sheet"],
        record["Cell"],
        record["Type"]
    )
    for record
    in PARENT_CELL_RECORDS
]

branch_c_signature = [
    (
        record["Sheet"],
        record["Cell"],
        record["Type"]
    )
    for record
    in BRANCH_C_CELL_RECORDS
]

cell_identity_type_and_order_preserved = (
    branch_c_signature
    == parent_signature
)


string_values_correctly_normalised = True
non_string_values_preserved = True
normalised_string_cell_count = 0

for parent_record, branch_c_record in zip(
    PARENT_CELL_RECORDS,
    BRANCH_C_CELL_RECORDS
):

    parent_value = parent_record[
        "Decoded Value"
    ]

    branch_c_value = branch_c_record[
        "Decoded Value"
    ]

    if isinstance(
        parent_value,
        str
    ):

        expected_value = (
            normalise_string_value(
                parent_value
            )
        )

        if (
            branch_c_value
            != expected_value
        ):
            string_values_correctly_normalised = False
            break

        if (
            parent_value
            != expected_value
        ):
            normalised_string_cell_count += 1

    else:

        if (
            branch_c_value
            != parent_value
        ):
            non_string_values_preserved = False
            break


branch_c_cell_key_set = {
    (
        record["Sheet"],
        record["Cell"]
    )
    for record
    in BRANCH_C_CELL_RECORDS
}

all_source_cells_preserved = (
    branch_c_cell_key_set
    == set(parent_cell_keys)
)

selected_scope_value_cells_preserved = all(
    (
        row["Sheet"],
        row["Expected Value Cell"]
    )
    in branch_c_cell_key_set
    for _, row
    in selection_scope_df.iterrows()
)

selected_scope_flag_cells_preserved_when_nonempty = all(
    (
        (
            row["Sheet"],
            row["Expected Flag Cell"]
        )
        in branch_c_cell_key_set
    )
    or
    (
        not row[
            "Expected Statistical Flag"
        ]
    )
    for _, row
    in selection_scope_df.iterrows()
)


# Exact reproducibility from the declared normalisation procedure.
reproduced_lines = []

for line in SOURCE_B_MARKDOWN.splitlines():

    cell_match = (
        CELL_LINE_PATTERN.fullmatch(
            line.strip()
        )
    )

    if cell_match:

        coordinate = cell_match.group(1)
        value_type = cell_match.group(2)
        decoded_value = json.loads(
            cell_match.group(3)
        )

        if isinstance(
            decoded_value,
            str
        ):
            decoded_value = (
                normalise_string_value(
                    decoded_value
                )
            )

        encoded_value = json.dumps(
            decoded_value,
            ensure_ascii=False,
            allow_nan=False
        )

        reproduced_lines.append(
            f"- {coordinate} "
            f"[{value_type}]: "
            f"{encoded_value}"
        )

    else:

        reproduced_lines.append(
            normalise_non_cell_line(
                line
            )
        )


REPRODUCED_NORMALISED_MARKDOWN = (
    "\n".join(
        reproduced_lines
    )
    .rstrip()
    + "\n"
)

deterministic_representation_verified = (
    REPRODUCED_NORMALISED_MARKDOWN
    == NORMALISED_MARKDOWN
)


normalisation_integrity_passed = bool(
    PARENT_EQUIVALENCE_PASSED
    and branch_c_cell_count_valid
    and worksheet_order_preserved
    and cell_identity_type_and_order_preserved
    and string_values_correctly_normalised
    and non_string_values_preserved
    and all_source_cells_preserved
    and selected_scope_value_cells_preserved
    and selected_scope_flag_cells_preserved_when_nonempty
    and deterministic_representation_verified
)


normalisation_check = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "expected_worksheet_count":
        EXPECTED_WORKSHEET_COUNT,

    "parent_worksheet_count":
        len(
            PARENT_WORKSHEET_ORDER
        ),

    "branch_C_worksheet_count":
        len(
            BRANCH_C_WORKSHEET_ORDER
        ),

    "worksheet_order_preserved":
        worksheet_order_preserved,

    "expected_non_empty_cell_count":
        EXPECTED_NON_EMPTY_CELL_COUNT,

    "parent_represented_cell_count":
        len(
            PARENT_CELL_RECORDS
        ),

    "branch_C_represented_cell_count":
        len(
            BRANCH_C_CELL_RECORDS
        ),

    "cell_count_preserved":
        branch_c_cell_count_valid,

    "cell_identity_type_and_order_preserved":
        cell_identity_type_and_order_preserved,

    "all_source_cells_preserved":
        all_source_cells_preserved,

    "string_values_correctly_normalised":
        string_values_correctly_normalised,

    "normalised_string_cell_count":
        normalised_string_cell_count,

    "non_string_values_preserved":
        non_string_values_preserved,

    "selected_scope_value_cells_preserved":
        selected_scope_value_cells_preserved,

    "selected_scope_flag_cells_preserved_when_nonempty":
        selected_scope_flag_cells_preserved_when_nonempty,

    "deterministic_representation_verified":
        deterministic_representation_verified,

    "complete_16_sheet_representation_retained":
        True,

    "complete_source_cell_population_retained":
        True,

    "worksheet_filtering_applied":
        False,

    "row_filtering_applied":
        False,

    "column_filtering_applied":
        False,

    "selected_scope_filtering_applied":
        False,

    "cell_reordering_applied":
        False,

    "unicode_nfkc_normalisation_applied":
        True,

    "unicode_space_standardisation_applied":
        True,

    "apostrophe_and_quote_standardisation_applied":
        True,

    "dash_and_minus_standardisation_applied":
        True,

    "soft_hyphen_removal_applied":
        True,

    "string_whitespace_normalisation_applied":
        True,

    "semantic_harmonisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "statistical_flag_reconstruction_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "numeric_rescaling_applied":
        False,

    "numeric_rounding_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_integrity_passed
}


NORMALISATION_CHECK_PATH.write_text(
    json.dumps(
        normalisation_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        normalisation_check,
        ensure_ascii=False,
        indent=2
    )
)


if not normalisation_integrity_passed:
    raise ValueError(
        "D13 Branch C normalisation-integrity checks failed."
    )


{
  "document_id": "D13",
  "branch": "C",
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "expected_worksheet_count": 16,
  "parent_worksheet_count": 16,
  "branch_C_worksheet_count": 16,
  "worksheet_order_preserved": true,
  "expected_non_empty_cell_count": 12978,
  "parent_represented_cell_count": 12978,
  "branch_C_represented_cell_count": 12978,
  "cell_count_preserved": true,
  "cell_identity_type_and_order_preserved": true,
  "all_source_cells_preserved": true,
  "string_values_correctly_normalised": true,
  "normalised_string_cell_count": 48,
  "non_string_values_preserved": true,
  "selected_scope_value_cells_preserved": true,
  "selected_scope_flag_cells_preserved_when_nonempty": true,
  "deterministic_representation_verified": true,
  "complete_16_sheet_representation_retained": true,
  "complete_source_cell_population_retained": true,
  "worksheet_filtering_applied": false,
  "row_filtering_applied": false,
  "column_filtering_applied": false,
  "selected_sc

In [10]:
# ============================================================
# 8. Save Branch C representation
# ============================================================

REPRESENTATION_PATH.write_text(
    NORMALISED_MARKDOWN,
    encoding="utf-8"
)

REPRESENTATION_SHA256 = sha256_file(
    REPRESENTATION_PATH
)

print(
    "Saved:",
    REPRESENTATION_PATH.name
)

print(
    "Representation SHA-256:",
    REPRESENTATION_SHA256
)


Saved: D13_branch_C_normalised_markdown.md
Representation SHA-256: 497768d6c94bdd9df4f720de9bf726f8f5302f87a6279e770fc7958924df7955


In [11]:
# ============================================================
# 9. Create controlled Branch C extraction prompt
# ============================================================
#
# The selected worksheets/geographies/years define the fixed task and
# are disclosed in every branch.
#
# Expected record counts, expected flags, exact values and expected
# source-cell identities are NOT disclosed to the model.
# ============================================================

sheet_lines = "\n".join(
    f"- {sheet_name}"
    for sheet_name in SELECTED_SHEETS
)

geography_lines = "\n".join(
    f"- {geography}"
    for geography in SELECTED_GEOGRAPHIES
)

year_lines = "\n".join(
    f"- {year}"
    for year in SELECTED_YEARS
)


BRANCH_C_PROMPT = f"""You are an information extraction assistant.

Extract the predefined unemployment-rate observations represented in
the attached deterministically normalised structural Markdown
representation of the workbook:

"Unemployment rates by country of birth"

Treat the attached deterministically normalised structural Markdown
representation as the only source of information.

The representation contains the complete original workbook and
preserves worksheet names, physical cell coordinates, source types and
source values.


Selected worksheets:

{sheet_lines}


Selected geographies:

{geography_lines}


Selected reporting years:

{year_lines}


For every represented combination of selected worksheet, selected
geography and selected reporting year, return one record.

For every included record return exactly these fields:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location


Use these fixed semantic values:

Category:
Labour market time series

Topic:
Unemployment rate by country of birth

Unit:
percent


Worksheet-level dimensions:

For each selected worksheet, read the represented worksheet-level
dimensions directly from the structural Markdown:

- Sex
- Age Class
- Country/Region of Birth

Do not infer these dimensions from the worksheet number or from another
worksheet.


Description:

Construct Description in exactly this order:

Geography: <geography>; Sex: <sex>; Age class: <age class>;
Country/region of birth: <birth category>

Use the dimension wording represented in the corresponding worksheet.

When the statistical-flag cell directly adjacent to the selected value
cell is non-empty, append:

; Statistical flag: <flag>

Do not append a Statistical flag segment when the adjacent flag cell
is blank.

Preserve the represented statistical flag exactly.
Do not infer, rewrite, expand or interpret statistical flags.


Value:

- Extract the numerical unemployment-rate value represented for the
  selected geography and selected reporting year.
- Use the value cell, not the adjacent statistical-flag cell.
- Return the value as a JSON number.
- Do not calculate, aggregate, interpolate, round, convert or correct
  the represented value.


Reporting Period:

- Use the selected reporting year as a four-digit string.
- Example:
  "2024"


Source Location:

- Identify the exact workbook cell coordinate represented for the
  numerical value.
- Use this form:
  "Sheet N, cell A1"
- Use the value cell, not the adjacent statistical-flag cell.
- Determine the cell directly from the structural Markdown.


Extraction rules:

- Use only the selected worksheets.
- Use only the selected geographies.
- Use only the selected reporting years.
- Return one observation for every represented combination within this
  predefined scope.
- Do not omit a required combination.
- Do not return observations outside the predefined scope.
- Read Sex, Age Class and Country/Region of Birth directly from each
  selected worksheet.
- Preserve exact geography and dimension labels.
- Preserve an adjacent statistical flag only when explicitly represented.
- Do not calculate missing observations.
- Do not aggregate values.
- Do not interpolate values.
- Do not convert percentages into another scale.
- Do not infer missing statistical flags.
- Do not use external knowledge.
- Do not use observations from unselected worksheets.
- Do not duplicate records.
- Ignore structural Markdown headings and source-type labels except as
  aids for locating worksheet, row and cell structure.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.


Expected JSON structure:

{{
  "document_id": "D13",
  "branch": "C",
  "records": [
    {{
      "Category": "Labour market time series",
      "Topic": "Unemployment rate by country of birth",
      "Description": null,
      "Value": null,
      "Unit": "percent",
      "Reporting Period": null,
      "Source Location": null
    }}
  ]
}}

Return only the JSON object.
"""


PROMPT_PATH.write_text(
    BRANCH_C_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)

print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)


Prompt saved: D13_branch_C_prompt.txt
Prompt SHA-256: ade22e035afaef5b3106fad6da65ff606c21acee4137f5bc2543e06218afcbbd


In [12]:
# ============================================================
# 10. Create representation and pre-extraction metadata
# ============================================================

REPRESENTATION_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "parent_B_representation_file":
        BRANCH_B_REPRESENTATION_PATH.name,

    "parent_B_representation_sha256":
        UPLOADED_BRANCH_B_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "representation_type":
        (
            "Complete frozen Branch B coordinate-aware structural "
            "Markdown workbook with deterministic normalisation"
        ),

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "complete_16_sheet_representation_retained":
        True,

    "represented_non_empty_cell_count":
        len(
            BRANCH_C_CELL_RECORDS
        ),

    "scope_enforced_by_prompt_not_representation_filtering":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "branch_B_regeneration_attempted":
        False,

    "normalisation_applied":
        True,

    "semantic_rewriting_applied":
        False,

    "statistical_flag_reconstruction_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "numeric_rescaling_applied":
        False,

    "numeric_rounding_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ]
}


REPRESENTATION_METADATA_PATH.write_text(
    json.dumps(
        REPRESENTATION_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_METADATA_PRE = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_INTEGRITY_VALID,

    "input_representation":
        (
            "Complete deterministically normalised "
            "coordinate-aware structural Markdown workbook"
        ),

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "direct_document_ingestion":
        False,

    "structural_conversion_applied":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "complete_source_workbook_retained":
        True,

    "worksheet_filtering_applied_to_model_input":
        False,

    "row_filtering_applied_to_model_input":
        False,

    "column_filtering_applied_to_model_input":
        False,

    "selected_scope_filtering_applied_to_model_input":
        False,

    "reference_values_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "reference_record_count_explicitly_disclosed_to_model":
        False,

    "reference_year_counts_explicitly_disclosed_to_model":
        False,

    "worksheet_dimension_answers_disclosed_to_model":
        False,

    "statistical_flag_answers_disclosed_to_model":
        False,

    "source_cell_reference_answers_disclosed_to_model":
        False,

    "selected_scope_disclosed_to_model":
        True,

    "selected_sheets":
        SELECTED_SHEETS,

    "selected_geographies":
        SELECTED_GEOGRAPHIES,

    "selected_years":
        SELECTED_YEARS,

    "manual_response_repair_permitted":
        False,

    "expected_output_format":
        "JSON object with document_id, branch and records",

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "execution_environment":
        "Independent ChatGPT conversation",

    "model":
        "GPT-5.5",

    "created_at":
        datetime.now().isoformat(),

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "validation_status":
        "Pending independent Branch C extraction and Stage 4 Validation C"
}


EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D13",
  "document_name": "Eurostat — Unemployment rates by country of birth",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D13 - Unemployment rates by country of birth_2026.xlsx",
  "source_sha256": "13f6d031c0e5c888d5632ad20853d16e603f462cdf3b58c0dd40057824d10f9a",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised coordinate-aware structural Markdown workbook",
  "representation_file": "D13_branch_C_normalised_markdown.md",
  "representation_sha256": "497768d6c94bdd9df4f720de9bf726f8f5302f87a6279e770fc7958924df7955",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "direct_document_ingestion": false,
  "structural_conversion_applied": true,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "complete_source_workbook_retained": true,
  "worksheet_filtering_applied_to_model_input": false,
  

In [13]:
# ============================================================
# 11. Final pre-extraction control check
# ============================================================

PRECHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_identity_verified":
        SOURCE_HASH_MATCH,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "complete_16_sheet_representation_retained":
        True,

    "worksheet_order_preserved":
        worksheet_order_preserved,

    "cell_identity_type_and_order_preserved":
        cell_identity_type_and_order_preserved,

    "all_source_cells_preserved":
        all_source_cells_preserved,

    "non_string_values_preserved":
        non_string_values_preserved,

    "selected_scope_value_cells_preserved":
        selected_scope_value_cells_preserved,

    "selected_scope_flag_cells_preserved_when_nonempty":
        selected_scope_flag_cells_preserved_when_nonempty,

    "representation_exists":
        REPRESENTATION_PATH.exists(),

    "prompt_exists":
        PROMPT_PATH.exists(),

    "expected_record_count_disclosed_to_model":
        False,

    "expected_year_counts_disclosed_to_model":
        False,

    "statistical_flag_answers_disclosed_to_model":
        False,

    "source_cell_reference_answers_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "ready_for_independent_llm_execution":
        bool(
            SOURCE_INTEGRITY_VALID
            and PARENT_EQUIVALENCE_PASSED
            and normalisation_check[
                "normalisation_integrity_passed"
            ]
            and REPRESENTATION_PATH.exists()
            and PROMPT_PATH.exists()
        )
}


PRECHECK_PATH.write_text(
    json.dumps(
        PRECHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        PRECHECK,
        ensure_ascii=False,
        indent=2
    )
)


if not PRECHECK[
    "ready_for_independent_llm_execution"
]:
    raise ValueError(
        "D13 Branch C is not ready for independent LLM execution."
    )


{
  "document_id": "D13",
  "branch": "C",
  "source_identity_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "complete_16_sheet_representation_retained": true,
  "worksheet_order_preserved": true,
  "cell_identity_type_and_order_preserved": true,
  "all_source_cells_preserved": true,
  "non_string_values_preserved": true,
  "selected_scope_value_cells_preserved": true,
  "selected_scope_flag_cells_preserved_when_nonempty": true,
  "representation_exists": true,
  "prompt_exists": true,
  "expected_record_count_disclosed_to_model": false,
  "expected_year_counts_disclosed_to_model": false,
  "statistical_flag_answers_disclosed_to_model": false,
  "source_cell_reference_answers_disclosed_to_model": false,
  "reference_values_used_for_transformation": false,
  "ready_for_independent_llm_execution": true
}


In [14]:
# ============================================================
# 12. Download pre-extraction Branch C artefacts
# ============================================================

for path in [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    SELECTION_SCOPE_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH
]:
    files.download(
        path
    )


print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D13_branch_C_normalised_markdown.md.\n"
    "3. Submit D13_branch_C_prompt.txt exactly once.\n"
    "4. Do not upload the original XLSX, Branch B artefacts, Stage 1 "
    "reference values, selection-scope diagnostics, or previous outputs.\n"
    "5. Do not manually repair, correct, reorder, deduplicate, or "
    "regenerate the response.\n"
    "6. Save the complete first response exactly as returned in TXT."
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Independent extraction instructions:
1. Open a new independent ChatGPT conversation.
2. Upload ONLY D13_branch_C_normalised_markdown.md.
3. Submit D13_branch_C_prompt.txt exactly once.
4. Do not upload the original XLSX, Branch B artefacts, Stage 1 reference values, selection-scope diagnostics, or previous outputs.
5. Do not manually repair, correct, reorder, deduplicate, or regenerate the response.
6. Save the complete first response exactly as returned in TXT.


In [15]:
# ============================================================
# 13. Upload and preserve the untouched Branch C response
# ============================================================

uploaded_response = files.upload()

if len(
    uploaded_response
) != 1:
    raise ValueError(
        "Upload exactly one complete raw D13 Branch C response file."
    )


RAW_RESPONSE_SOURCE = Path(
    next(
        iter(
            uploaded_response
        )
    )
)


RAW_RESPONSE_TEXT = (
    RAW_RESPONSE_SOURCE.read_text(
        encoding="utf-8"
    )
)

if not RAW_RESPONSE_TEXT.strip():
    raise ValueError(
        "Uploaded D13 Branch C response is empty."
    )


RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)


print(
    "Raw response preserved unchanged."
)

print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)


Saving D13_branch_C_raw_response.txt to D13_branch_C_raw_response.txt
Raw response preserved unchanged.
Raw response SHA-256: 15cf46465599c450ee856a033f9d96e90358fcf1ded5a808fe3af8687d17d68d


In [16]:
# ============================================================
# 14. Parse raw response WITHOUT repair
# ============================================================

valid_json = True
json_parsing_error = None
parsed_response = None


try:
    parsed_response = json.loads(
        RAW_RESPONSE_TEXT
    )

except json.JSONDecodeError as exc:
    valid_json = False
    json_parsing_error = str(
        exc
    )


top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)

document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_response
)

document_id_correct = (
    document_id_present
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_response
)

branch_correct = (
    branch_present
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)

records_present = (
    top_level_object_valid
    and "records"
    in parsed_response
)

records_is_list = (
    records_present
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)

records_evaluable = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
)

extracted_records = (
    parsed_response[
        "records"
    ]
    if records_evaluable
    else []
)

observed_record_count = (
    len(
        extracted_records
    )
    if records_evaluable
    else None
)


print(
    "Valid JSON:",
    valid_json
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed records:",
    observed_record_count
)

if json_parsing_error:
    print(
        "JSON parsing error:",
        json_parsing_error
    )


Valid JSON: True
Records evaluable: True
Observed records: 75


In [17]:
# ============================================================
# 15. Validate record schema and field types
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []


if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):
            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Record is not a JSON object"
            })

            continue


        observed_fields = list(
            record.keys()
        )

        if observed_fields != EXPECTED_FIELDS:

            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Field names or field order differ",

                "expected_fields":
                    EXPECTED_FIELDS,

                "observed_fields":
                    observed_fields
            })


        for field in STRING_OR_NULL_FIELDS:

            value = record.get(
                field
            )

            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):

                field_type_issues.append({
                    "record_index":
                        record_index,

                    "field":
                        field,

                    "observed_type":
                        type(
                            value
                        ).__name__
                })


        value = record.get(
            "Value"
        )

        if (
            value is not None
            and (
                isinstance(
                    value,
                    bool
                )
                or not isinstance(
                    value,
                    (int, float)
                )
            )
        ):

            field_type_issues.append({
                "record_index":
                    record_index,

                "field":
                    "Value",

                "observed_type":
                    type(
                        value
                    ).__name__
            })


        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(
                field
            )

            if (
                value is None
                or (
                    isinstance(
                        value,
                        str
                    )
                    and not value.strip()
                )
            ):

                missing_mandatory_values.append({
                    "record_index":
                        record_index,

                    "field":
                        field
                })


record_schema_valid = (
    len(
        record_structure_issues
    )
    == 0
    if records_evaluable
    else None
)

field_types_valid = (
    len(
        field_type_issues
    )
    == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(
        missing_mandatory_values
    )
    == 0
    if records_evaluable
    else None
)


print(
    "Record schema valid:",
    record_schema_valid
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Mandatory fields complete:",
    mandatory_fields_complete
)


Record schema valid: True
Field types valid: True
Mandatory fields complete: True


In [18]:
# ============================================================
# 16. D13 content/scope diagnostics kept separate from schema validity
# ============================================================

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    observed_category_counts = dict(
        Counter(
            record.get(
                "Category"
            )
            for record
            in extracted_records
            if isinstance(
                record,
                dict
            )
        )
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


    observed_year_counts = dict(
        Counter(
            str(
                record.get(
                    "Reporting Period"
                )
            ).strip()
            for record
            in extracted_records
            if isinstance(
                record,
                dict
            )
        )
    )


    year_counts_valid = (
        observed_year_counts
        == EXPECTED_YEAR_COUNTS
    )


    category_constant_valid = all(
        record.get(
            "Category"
        )
        == EXPECTED_CATEGORY
        for record
        in extracted_records
        if isinstance(
            record,
            dict
        )
    )


    topic_constant_valid = all(
        record.get(
            "Topic"
        )
        == EXPECTED_TOPIC
        for record
        in extracted_records
        if isinstance(
            record,
            dict
        )
    )


    unit_constant_valid = all(
        record.get(
            "Unit"
        )
        == EXPECTED_UNIT
        for record
        in extracted_records
        if isinstance(
            record,
            dict
        )
    )


    constant_fields_valid = all([
        category_constant_valid,
        topic_constant_valid,
        unit_constant_valid
    ])


    DESCRIPTION_LABELS = [
        "Geography:",
        "Sex:",
        "Age class:",
        "Country/region of birth:"
    ]


    description_dimension_labels_valid = all(
        isinstance(
            record.get(
                "Description"
            ),
            str
        )
        and all(
            label
            in record.get(
                "Description",
                ""
            )
            for label
            in DESCRIPTION_LABELS
        )
        for record
        in extracted_records
        if isinstance(
            record,
            dict
        )
    )


    observed_flag_segment_count = sum(
        1
        for record
        in extracted_records
        if (
            isinstance(
                record,
                dict
            )
            and isinstance(
                record.get(
                    "Description"
                ),
                str
            )
            and "Statistical flag:"
            in record[
                "Description"
            ]
        )
    )


    source_location_pattern = re.compile(
        r"^Sheet [1-5], cell [A-Z]+\d+$"
    )


    observed_source_locations = [
        record.get(
            "Source Location"
        )
        for record
        in extracted_records
        if isinstance(
            record,
            dict
        )
    ]


    source_location_format_valid = all(
        isinstance(
            location,
            str
        )
        and source_location_pattern.fullmatch(
            location.strip()
        )
        is not None
        for location
        in observed_source_locations
    )


    observed_source_location_set = set(
        observed_source_locations
    )


    expected_source_locations_complete = (
        observed_source_location_set
        == expected_source_location_set
    )


    extracted_source_locations_unique = (
        len(
            observed_source_locations
        )
        == len(
            observed_source_location_set
        )
    )


    unexpected_source_locations = sorted(
        observed_source_location_set
        - expected_source_location_set
    )


    missing_source_locations = sorted(
        expected_source_location_set
        - observed_source_location_set
    )


    fixed_scope_valid = all([
        expected_source_locations_complete,
        extracted_source_locations_unique
    ])


    duplicate_complete_record_signature_count = sum(
        1
        for count
        in Counter(
            tuple(
                json.dumps(
                    record.get(
                        field
                    ),
                    ensure_ascii=False,
                    sort_keys=True
                )
                for field
                in EXPECTED_FIELDS
            )
            for record
            in extracted_records
            if isinstance(
                record,
                dict
            )
        ).values()
        if count > 1
    )


else:

    record_count_valid = None
    observed_category_counts = None
    category_counts_valid = None
    observed_year_counts = None
    year_counts_valid = None
    category_constant_valid = None
    topic_constant_valid = None
    unit_constant_valid = None
    constant_fields_valid = None
    description_dimension_labels_valid = None
    observed_flag_segment_count = None
    observed_source_locations = None
    source_location_format_valid = None
    expected_source_locations_complete = None
    extracted_source_locations_unique = None
    unexpected_source_locations = None
    missing_source_locations = None
    fixed_scope_valid = None
    duplicate_complete_record_signature_count = None


scope_complete = (
    all([
        record_count_valid is True,
        category_counts_valid is True,
        year_counts_valid is True,
        fixed_scope_valid is True
    ])
    if records_evaluable
    else False
)


CONTENT_DIAGNOSTICS = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches_reference":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match_reference":
        category_counts_valid,

    "expected_year_counts":
        EXPECTED_YEAR_COUNTS,

    "observed_year_counts":
        observed_year_counts,

    "year_counts_match_reference":
        year_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        (
            len(
                missing_mandatory_values
            )
            if records_evaluable
            else None
        ),

    "constant_fields_valid":
        constant_fields_valid,

    "description_dimension_labels_valid":
        description_dimension_labels_valid,

    "expected_flagged_record_count":
        EXPECTED_FLAGGED_RECORDS,

    "observed_flag_segment_count":
        observed_flag_segment_count,

    "flag_segment_count_matches_reference":
        (
            observed_flag_segment_count
            == EXPECTED_FLAGGED_RECORDS
            if records_evaluable
            else None
        ),

    "source_location_format_valid":
        source_location_format_valid,

    "expected_source_locations_complete":
        expected_source_locations_complete,

    "source_locations_unique":
        extracted_source_locations_unique,

    "unexpected_source_locations":
        unexpected_source_locations,

    "missing_source_locations":
        missing_source_locations,

    "fixed_scope_valid":
        fixed_scope_valid,

    "duplicate_complete_record_signature_count":
        duplicate_complete_record_signature_count
}


scope_check_rows = []

if records_evaluable:

    for expected_location in sorted(
        expected_source_location_set
    ):

        occurrence_count = (
            observed_source_locations.count(
                expected_location
            )
        )

        scope_check_rows.append({
            "Expected Source Location":
                expected_location,

            "Observed Count":
                occurrence_count,

            "Valid":
                occurrence_count == 1
        })


pd.DataFrame(
    scope_check_rows
).to_csv(
    SCOPE_CHECK_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


{
  "expected_record_count": 75,
  "observed_record_count": 75,
  "record_count_matches_reference": true,
  "expected_category_counts": {
    "Labour market time series": 75
  },
  "observed_category_counts": {
    "Labour market time series": 75
  },
  "category_counts_match_reference": true,
  "expected_year_counts": {
    "2020": 25,
    "2022": 25,
    "2024": 25
  },
  "observed_year_counts": {
    "2020": 25,
    "2022": 25,
    "2024": 25
  },
  "year_counts_match_reference": true,
  "mandatory_fields_complete": true,
  "missing_mandatory_value_count": 0,
  "constant_fields_valid": true,
  "description_dimension_labels_valid": true,
  "expected_flagged_record_count": 17,
  "observed_flag_segment_count": 17,
  "flag_segment_count_matches_reference": true,
  "source_location_format_valid": true,
  "expected_source_locations_complete": true,
  "source_locations_unique": true,
  "unexpected_source_locations": [],
  "missing_source_locations": [],
  "fixed_scope_valid": true,
  "dupl

In [19]:
# ============================================================
# 17. Determine technical/schema validity
# ============================================================
#
# IMPORTANT:
# Counts, year distributions, fixed-scope completeness, Description
# structure and flag diagnostics are NOT conditions for technical
# schema validity.
# ============================================================

structure_valid = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
    and record_schema_valid is True
    and field_types_valid is True
)


STRUCTURE_CHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        (
            "Complete deterministically normalised "
            "coordinate-aware structural Markdown workbook"
        ),

    "valid_json":
        bool(
            valid_json
        ),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(
            top_level_object_valid
        ),

    "document_id_correct":
        bool(
            document_id_correct
        ),

    "branch_correct":
        bool(
            branch_correct
        ),

    "records_is_list":
        bool(
            records_is_list
        ),

    "records_evaluable":
        bool(
            records_evaluable
        ),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structure_valid":
        bool(
            structure_valid
        ),

    "scope_complete":
        bool(
            scope_complete
        )
}


STRUCTURE_CHECK_PATH.write_text(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D13",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised coordinate-aware structural Markdown workbook",
  "valid_json": true,
  "json_parsing_error": null,
  "top_level_object_valid": true,
  "document_id_correct": true,
  "branch_correct": true,
  "records_is_list": true,
  "records_evaluable": true,
  "record_schema_valid": true,
  "record_structure_issues": [],
  "field_types_valid": true,
  "field_type_issues": [],
  "content_diagnostics": {
    "expected_record_count": 75,
    "observed_record_count": 75,
    "record_count_matches_reference": true,
    "expected_category_counts": {
      "Labour market time series": 75
    },
    "observed_category_counts": {
      "Labour market time series": 75
    },
    "category_counts_match_reference": true,
    "expected_year_counts": {
      "2020": 25,
      "2022": 25,
      "2024": 25
    },
    "observed_year_counts": {
      "2020": 25,

In [20]:
# ============================================================
# 18. Preserve parsed extraction only when records are evaluable
# ============================================================

parsed_extraction_created = False
parsed_extraction_sha256 = None


if records_evaluable:

    canonical_extraction = {
        "document_id":
            DOCUMENT_ID,

        "branch":
            BRANCH,

        "records":
            extracted_records
    }


    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            canonical_extraction,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    parsed_extraction_sha256 = (
        sha256_file(
            PARSED_EXTRACTION_PATH
        )
    )

    parsed_extraction_created = True


    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH.name
    )

else:

    print(
        "No parsed extraction created because the preserved raw response "
        "does not contain an evaluable JSON records structure."
    )


Parsed extraction saved: D13_branch_C_parsed_extraction.json


In [21]:
# ============================================================
# 19. Create final experiment metadata and summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "observed_year_counts":
        observed_year_counts,

    "structure_check_file":
        STRUCTURE_CHECK_PATH.name,

    "scope_check_file":
        SCOPE_CHECK_PATH.name,

    "structure_valid":
        bool(
            structure_valid
        ),

    "notes":
        (
            "D13 Branch C applies deterministic non-semantic normalisation "
            "to the exact frozen Branch B complete 16-sheet coordinate-aware "
            "Markdown representation. All 12,978 represented non-empty source "
            "cells, worksheet order, cell coordinates, source types and "
            "non-string values are retained. No source-scope filtering, "
            "statistical-flag reconstruction, semantic rewriting, numerical "
            "conversion, rescaling, rounding or manual correction is applied. "
            "Expected Stage 1 answers and source-cell identities are not "
            "disclosed to the model. Accuracy is evaluated separately in "
            "Stage 4 Validation C."
        )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_INTEGRITY_VALID,

    "input_representation":
        (
            "Complete deterministically normalised "
            "coordinate-aware structural Markdown workbook"
        ),

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "structural_conversion_inherited_from_branch_B":
        True,

    "branch_B_regeneration_attempted":
        False,

    "normalisation_applied":
        True,

    "complete_source_workbook_retained":
        True,

    "source_worksheet_count":
        len(
            PARENT_WORKSHEET_ORDER
        ),

    "represented_non_empty_cell_count":
        len(
            BRANCH_C_CELL_RECORDS
        ),

    "selected_sheet_count":
        len(
            SELECTED_SHEETS
        ),

    "selected_geography_count":
        len(
            SELECTED_GEOGRAPHIES
        ),

    "selected_year_count":
        len(
            SELECTED_YEARS
        ),

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_year_counts_disclosed_to_model":
        False,

    "statistical_flag_answers_disclosed_to_model":
        False,

    "source_cell_reference_answers_disclosed_to_model":
        False,

    "raw_response_preserved":
        True,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "valid_json":
        bool(
            valid_json
        ),

    "records_evaluable":
        bool(
            records_evaluable
        ),

    "record_schema_valid":
        record_schema_valid,

    "field_types_valid":
        field_types_valid,

    "structure_valid":
        bool(
            structure_valid
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "expected_year_counts":
        EXPECTED_YEAR_COUNTS,

    "observed_year_counts":
        observed_year_counts,

    "year_counts_match":
        year_counts_valid,

    "constant_fields_valid":
        constant_fields_valid,

    "description_dimension_labels_valid":
        description_dimension_labels_valid,

    "expected_flagged_record_count":
        EXPECTED_FLAGGED_RECORDS,

    "observed_flag_segment_count":
        observed_flag_segment_count,

    "source_location_format_valid":
        source_location_format_valid,

    "expected_source_locations_complete":
        expected_source_locations_complete,

    "source_locations_unique":
        extracted_source_locations_unique,

    "scope_complete":
        bool(
            scope_complete
        ),

    "duplicate_complete_record_signature_count":
        duplicate_complete_record_signature_count,

    "parsed_extraction_created":
        parsed_extraction_created,

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "accuracy_validation_completed":
        False,

    "validation_status":
        (
            "Pending Stage 4 Branch C validation against the fixed Stage 1 "
            "reference dataset using Branch A-frozen D13 comparison rules"
            if records_evaluable
            else
            "Not content-evaluable because the preserved Branch C response "
            "does not contain an evaluable JSON records structure"
        )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D13",
  "document_name": "Eurostat — Unemployment rates by country of birth",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D13 - Unemployment rates by country of birth_2026.xlsx",
  "source_sha256": "13f6d031c0e5c888d5632ad20853d16e603f462cdf3b58c0dd40057824d10f9a",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised coordinate-aware structural Markdown workbook",
  "representation_file": "D13_branch_C_normalised_markdown.md",
  "representation_sha256": "497768d6c94bdd9df4f720de9bf726f8f5302f87a6279e770fc7958924df7955",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "structural_conversion_inherited_from_branch_B": true,
  "branch_B_regeneration_attempted": false,
  "normalisation_applied": true,
  "complete_source_workbook_retained": true,
  "source_worksheet_count": 16,
  "represented_non_empty_cell_count": 12978,
  "selected_she

In [22]:
# ============================================================
# 20. Final artefact inventory and downloads
# ============================================================

artefacts = [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    SELECTION_SCOPE_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH,
    RAW_RESPONSE_PATH,
    STRUCTURE_CHECK_PATH,
    SCOPE_CHECK_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    artefacts.append(
        PARSED_EXTRACTION_PATH
    )


print(
    "Final D13 Branch C artefacts:"
)

for path in artefacts:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )


for path in artefacts:

    if path.exists():

        files.download(
            path
        )


Final D13 Branch C artefacts:
- D13_branch_C_parent_B_equivalence_check.json | exists: True
- D13_branch_C_normalisation_check.json | exists: True
- D13_branch_C_normalised_markdown.md | exists: True
- D13_branch_C_representation_metadata.json | exists: True
- D13_branch_C_selection_scope.csv | exists: True
- D13_branch_C_prompt.txt | exists: True
- D13_branch_C_experiment_metadata_pre.json | exists: True
- D13_branch_C_pre_extraction_check.json | exists: True
- D13_branch_C_raw_response.txt | exists: True
- D13_branch_C_structure_check.json | exists: True
- D13_branch_C_scope_check.csv | exists: True
- D13_branch_C_experiment_metadata.json | exists: True
- D13_branch_C_experiment_summary.json | exists: True
- D13_branch_C_parsed_extraction.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>